# Unidad 2 · Colab 1 de 3
## Manejo de excepciones y código limpio

**Objetivos de este notebook**

- Manejar errores de forma estructurada con `try`/`except`/`else`/`finally`.
- Crear excepciones personalizadas y saber cuándo tiene sentido usarlas.
- Re-lanzar excepciones preservando contexto (`raise ... from ...`).
- Aplicar PEP 8, docstrings y type hinting para escribir código más claro y mantenible.

> **Nivel:** intermedio. Se asume Python básico (funciones, clases).

---

## 1. `try` / `except` / `else` / `finally`

```python
try:
    resultado = 10 / 0
except ZeroDivisionError as e:
    print('Error:', e)
else:
    print('Sin errores, resultado:', resultado)
finally:
    print('Esto se ejecuta siempre, haya error o no')
```

- `try`: el código que puede fallar.
- `except`: qué hacer si falla (podés tener varios, para distintos tipos de error).
- `else`: se ejecuta solo si no hubo excepción.
- `finally`: se ejecuta siempre, haya error o no (ideal para liberar recursos: cerrar archivos, conexiones).

Documentación oficial: [Errores y excepciones (tutorial de Python)](https://docs.python.org/3/tutorial/errors.html) · [Excepciones incorporadas](https://docs.python.org/3/library/exceptions.html)

In [ ]:
def dividir_seguro(a, b):
    try:
        return a / b
    except ZeroDivisionError:
        print('No se puede dividir por cero')
        return None
    except TypeError:
        print('Los argumentos deben ser numeros')
        return None

print(dividir_seguro(10, 2))
print(dividir_seguro(10, 0))
print(dividir_seguro(10, 'a'))

### Ejercicio 1 — Múltiples `except`

Escribí una función `obtener_elemento(lista, indice)` que devuelva `lista[indice]`, capturando `IndexError` (índice fuera de rango) y `TypeError` (índice no es un entero), imprimiendo un mensaje distinto para cada caso y devolviendo `None`.

<details>
<summary>💡 Ver solución</summary>

```python
def obtener_elemento(lista, indice):
    try:
        return lista[indice]
    except IndexError:
        print('Indice fuera de rango')
        return None
    except TypeError:
        print('El indice debe ser un entero')
        return None

obtener_elemento([1, 2, 3], 10)
obtener_elemento([1, 2, 3], 'a')
```

</details>

## 2. Excepciones personalizadas

Cuando el error es específico de tu dominio (no un error genérico de Python), conviene crear tu propia excepción — hace el código más legible y permite capturar justo ese caso.

```python
class SaldoInsuficienteError(Exception):
    def __init__(self, saldo, monto):
        self.saldo = saldo
        self.monto = monto
        super().__init__(f'Saldo {saldo} insuficiente para retirar {monto}')

def retirar(saldo, monto):
    if monto > saldo:
        raise SaldoInsuficienteError(saldo, monto)
    return saldo - monto

try:
    retirar(100, 500)
except SaldoInsuficienteError as e:
    print('No se pudo retirar:', e)
```

### Ejercicio 2 — Crear una excepción propia

Creá una excepción `MontoInvalidoError(Exception)` y una función `validar_monto(monto)` que la lance si `monto <= 0`, con un mensaje que incluya el monto recibido. Probala con un monto negativo dentro de un `try`/`except`.

<details>
<summary>💡 Ver solución</summary>

```python
class MontoInvalidoError(Exception):
    pass

def validar_monto(monto):
    if monto <= 0:
        raise MontoInvalidoError(f'El monto {monto} debe ser mayor a cero')
    return monto

try:
    validar_monto(-50)
except MontoInvalidoError as e:
    print('Error de validacion:', e)
```

</details>

## 3. Re-lanzar excepciones con contexto (`raise ... from ...`)

A veces atrapás un error de bajo nivel y querés lanzar uno más específico para quien llama, sin perder el error original:

```python
def leer_configuracion(texto):
    import json
    try:
        return json.loads(texto)
    except json.JSONDecodeError as e:
        raise ValueError('Configuracion invalida, no es JSON valido') from e
```

`from e` conserva la traza original (útil para debuggear), pero deja claro cuál es el error relevante para quien usa la función.

### Ejercicio 3 — Encadenar excepciones

Escribí una función `cargar_precio(texto)` que intente convertir `texto` a `float` con `float(texto)`, y si falla (`ValueError`), lance una `MontoInvalidoError` (la del Ejercicio 2) encadenada con `from e`, con un mensaje claro.

<details>
<summary>💡 Ver solución</summary>

```python
def cargar_precio(texto):
    try:
        return float(texto)
    except ValueError as e:
        raise MontoInvalidoError(f'No se pudo interpretar {texto} como un monto') from e

try:
    cargar_precio('abc')
except MontoInvalidoError as e:
    print(e)
    print('Causa original:', e.__cause__)
```

</details>

## 4. PEP 8: estilo de código

[PEP 8](https://peps.python.org/pep-0008/) es la guía de estilo oficial de Python. Algunos puntos clave:

- Nombres de variables y funciones en `snake_case`; clases en `PascalCase`; constantes en `MAYUSCULAS`.
- Líneas de hasta 79-99 caracteres (según el proyecto).
- Dos líneas en blanco entre definiciones de funciones/clases a nivel de módulo.
- Espacios alrededor de operadores (`x = 1`, no `x=1`).

En la práctica, casi nadie revisa esto a mano: se usan herramientas como [ruff](https://docs.astral.sh/ruff/) o [black](https://black.readthedocs.io/en/stable/) para formatear y chequear el código automáticamente (incluso podés integrarlas al workflow de CI de la Unidad 6).

## 5. Docstrings y type hinting

```python
def calcular_descuento(precio: float, porcentaje: float) -> float:
    """Calcula el precio final aplicando un descuento porcentual.

    Args:
        precio: precio original del producto.
        porcentaje: descuento a aplicar, entre 0 y 100.

    Returns:
        El precio final con el descuento aplicado.

    Raises:
        ValueError: si el porcentaje no esta entre 0 y 100.
    """
    if not 0 <= porcentaje <= 100:
        raise ValueError('El porcentaje debe estar entre 0 y 100')
    return precio * (1 - porcentaje / 100)
```

Los **type hints** (`precio: float`, `-> float`) no cambian el comportamiento en tiempo de ejecución, pero ayudan al editor a detectar errores y sirven de documentación viva. Se pueden chequear con herramientas como [mypy](https://mypy.readthedocs.io/en/stable/).

Documentación oficial: [PEP 257 (docstrings)](https://peps.python.org/pep-0257/) · [módulo typing](https://docs.python.org/3/library/typing.html)

### Ejercicio 4 — Agregar tipado y docstring

Esta función no tiene type hints ni docstring. Agregaselos, incluyendo una sección `Raises` si corresponde:

```python
def aplicar_impuesto(monto, tasa):
    if tasa < 0:
        raise ValueError('La tasa no puede ser negativa')
    return monto * (1 + tasa)
```

<details>
<summary>💡 Ver solución</summary>

```python
def aplicar_impuesto(monto: float, tasa: float) -> float:
    """Aplica una tasa de impuesto sobre un monto.

    Args:
        monto: monto base sobre el que se aplica el impuesto.
        tasa: tasa de impuesto, como fraccion (ej. 0.21 para 21%).

    Returns:
        El monto con el impuesto aplicado.

    Raises:
        ValueError: si la tasa es negativa.
    """
    if tasa < 0:
        raise ValueError('La tasa no puede ser negativa')
    return monto * (1 + tasa)
```

</details>

## Mini-proyecto: validar una transacción

Escribí una función `validar_transaccion(monto: float, moneda: str) -> None` que:

1. Lance una excepción personalizada `MontoInvalidoError` si `monto <= 0`.
2. Lance una excepción personalizada `MonedaInvalidaError` si `moneda` no está en `{'ARS', 'USD', 'EUR'}`.
3. Tenga type hints completos y un docstring con `Args` y `Raises`.
4. Probala con al menos 3 casos: uno válido, uno con monto inválido, uno con moneda inválida — capturando cada excepción por separado.

**Entregable:** las dos excepciones + la función + las 3 pruebas.

---

**Seguís en:** *Colab 2 — Módulos, paquetes y entornos virtuales*